<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Points---visualize-cProfile-using-snakeviz" data-toc-modified-id="Points---visualize-cProfile-using-snakeviz-1">Points - visualize cProfile using snakeviz</a></span><ul class="toc-item"><li><span><a href="#Write-memprofile-to-disk---from_geodataframe()" data-toc-modified-id="Write-memprofile-to-disk---from_geodataframe()-1.1">Write memprofile to disk - from_geodataframe()</a></span></li><li><span><a href="#Write-memprofile-to-disk---from_geodataframe2()" data-toc-modified-id="Write-memprofile-to-disk---from_geodataframe2()-1.2">Write memprofile to disk - from_geodataframe2()</a></span></li><li><span><a href="#from_geodataframe()" data-toc-modified-id="from_geodataframe()-1.3">from_geodataframe()</a></span></li><li><span><a href="#from_geodataframe2()" data-toc-modified-id="from_geodataframe2()-1.4">from_geodataframe2()</a></span></li><li><span><a href="#Line-profiling---using-from_geodataframe()" data-toc-modified-id="Line-profiling---using-from_geodataframe()-1.5">Line profiling - using from_geodataframe()</a></span></li><li><span><a href="#Line-profiling---from_geodataframe2()" data-toc-modified-id="Line-profiling---from_geodataframe2()-1.6">Line profiling - from_geodataframe2()</a></span></li><li><span><a href="#Memory-profiling---from_geodataframe()" data-toc-modified-id="Memory-profiling---from_geodataframe()-1.7">Memory profiling - from_geodataframe()</a></span></li><li><span><a href="#Memory-profiling---from_geodataframe2()" data-toc-modified-id="Memory-profiling---from_geodataframe2()-1.8">Memory profiling - from_geodataframe2()</a></span></li></ul></li></ul></div>

This notebook uses cProfiler, line_profiler, memory_profiler libraries to profile the efficiency of two different approaches to convert a GeoDataFrame to SeDF.

Notebook needs a couple of additional libraries to profile and read profile output files. Run these once in the new environment.

In [ ]:
! conda install snakeviz -y

In [ ]:
! conda install line_profiler

In [7]:
## uncomment this if you are testing a non-installed version of the Python API and update the path
# ##
# import sys
# sys.path.insert(0, r"E:\\code\\geosaurus\\geosaurus\\src")
# import os

In [1]:
import arcgis
print(arcgis.__file__)
import pandas as pd
from arcgis.features import GeoAccessor
import geopandas as gpd
import matplotlib.pyplot as plt
import os

%matplotlib inline

e:\code\geosaurus\geosaurus\src\arcgis\__init__.py


In [2]:
# load snakeviz
%load_ext snakeviz

In [3]:
data_path = r'\\qalab_server\pydata\v108\geosaurus\data_prep\spatial_ref_tests'
large_data_path = r'\\qalab_server\pydata\v108\geosaurus\Esri_Geoanalytics_datasets'

In [4]:
from arcgis.gis import GIS
gis = GIS()

## Points - visualize cProfile using snakeviz

In [5]:
%%time
file_path = os.path.join(data_path, 'points_gcs_nad83.shp')
geo_df = gpd.read_file(file_path)

Wall time: 115 ms


In [6]:
%%snakeviz
sedf = pd.DataFrame.spatial.from_geodataframe(geo_df)

 
*** Profile stats marshalled to file 'C:\\Users\\atma6951\\AppData\\Local\\Temp\\tmp56tyfi59'. 
Embedding SnakeViz in this document...


In [7]:
%%snakeviz
sedf2 = pd.DataFrame.spatial.from_geodataframe2(geo_df)

 
*** Profile stats marshalled to file 'C:\\Users\\atma6951\\AppData\\Local\\Temp\\tmpoyl8fldv'. 
Embedding SnakeViz in this document...


# Points - Memory profiling

In [8]:
%load_ext memory_profiler

In [22]:
%%memit
file_path = os.path.join(data_path, 'points_gcs_nad83.shp')
geo_df = gpd.read_file(file_path)

peak memory: 137.11 MiB, increment: 0.00 MiB


In [10]:
%%memit
sedf = pd.DataFrame.spatial.from_geodataframe(geo_df)

peak memory: 136.82 MiB, increment: 1.48 MiB


In [11]:
%%memit
sedf2 = pd.DataFrame.spatial.from_geodataframe2(geo_df)

peak memory: 136.82 MiB, increment: 0.00 MiB


### Write memprofile to disk - from_geodataframe()

In [23]:
%mprun -T mprof1 -f pd.DataFrame.spatial.from_geodataframe pd.DataFrame.spatial.from_geodataframe(geo_df)
print(open('mprof1', 'r').read())



*** Profile printout saved to text file mprof1. 
Filename: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py

Line #    Mem usage    Increment   Line Contents
  2700    137.1 MiB    137.1 MiB       @staticmethod
  2701                                 def from_geodataframe(geo_df):
  2702                                     """
  2703                                     Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2704                                     Requires geopandas library be installed in current environment.
  2705                             
  2706                                     =====================  ===============================================================
  2707                                     **Argument**           **Description**
  2708                                     ---------------------  ---------------------------------------------------------------
  2709                                     geo_df   

### Write memprofile to disk - from_geodataframe2()

In [20]:
%mprun -T mprof2 -f pd.DataFrame.spatial.from_geodataframe2 pd.DataFrame.spatial.from_geodataframe2(geo_df)



*** Profile printout saved to text file mprof2. 


In [21]:
print(open('mprof2', 'r').read())

Filename: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py

Line #    Mem usage    Increment   Line Contents
  2751    137.1 MiB    137.1 MiB       @staticmethod
  2752                                 def from_geodataframe2(geo_df):
  2753                                     """
  2754                                     Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2755                                     Requires geopandas library be installed in current environment.
  2756                             
  2757                                     =====================  ===============================================================
  2758                                     **Argument**           **Description**
  2759                                     ---------------------  ---------------------------------------------------------------
  2760                                     geo_df                 GeoDataFrame object, created using G

# Lines - line_profiler

In [25]:
file_path = os.path.join(data_path, 'lines_pcs_wgs84_utm_z15n.shp')
geo_df = gpd.read_file(file_path)

In [15]:
%load_ext line_profiler

### from_geodataframe()

In [24]:
%lprun -T lprof1 -f pd.DataFrame.spatial.from_geodataframe pd.DataFrame.spatial.from_geodataframe(geo_df)
print(open('lprof1', 'r').read())


*** Profile printout saved to text file 'lprof1'. 
Timer unit: 2.93249e-07 s

Total time: 0.0959359 s
File: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py
Function: from_geodataframe at line 2700

Line #      Hits         Time  Per Hit   % Time  Line Contents
  2700                                               @staticmethod
  2701                                               def from_geodataframe(geo_df):
  2702                                                   """
  2703                                                   Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2704                                                   Requires geopandas library be installed in current environment.
  2705                                           
  2706                                                   =====================  ===============================================================
  2707                                                   **Argu

### from_geodataframe2()

In [26]:
%lprun -T lprof2 -f pd.DataFrame.spatial.from_geodataframe2 pd.DataFrame.spatial.from_geodataframe2(geo_df)
print(open('lprof2', 'r').read())


*** Profile printout saved to text file 'lprof2'. 
Timer unit: 2.93249e-07 s

Total time: 0.281513 s
File: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py
Function: from_geodataframe2 at line 2751

Line #      Hits         Time  Per Hit   % Time  Line Contents
  2751                                               @staticmethod
  2752                                               def from_geodataframe2(geo_df):
  2753                                                   """
  2754                                                   Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2755                                                   Requires geopandas library be installed in current environment.
  2756                                           
  2757                                                   =====================  ===============================================================
  2758                                                   **Arg

# Moderate sized data

In [28]:
%%time
file_path = os.path.join(data_path,"..","large_files", "points_270krows_800mb_gcs.shp")
geo_df = gpd.read_file(file_path)

Wall time: 25.3 s


### Line profiling - using from_geodataframe()

In [29]:
%lprun -T lprof_800mb_meth1 -f pd.DataFrame.spatial.from_geodataframe pd.DataFrame.spatial.from_geodataframe(geo_df)
print(open('lprof_800mb_meth1', 'r').read())


*** Profile printout saved to text file 'lprof_800mb_meth1'. 
Timer unit: 2.93249e-07 s

Total time: 56.3142 s
File: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py
Function: from_geodataframe at line 2700

Line #      Hits         Time  Per Hit   % Time  Line Contents
  2700                                               @staticmethod
  2701                                               def from_geodataframe(geo_df):
  2702                                                   """
  2703                                                   Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2704                                                   Requires geopandas library be installed in current environment.
  2705                                           
  2706                                                   =====================  ===============================================================
  2707                                                

### Line profiling - from_geodataframe2()

In [31]:
%lprun -T lprof_800mb_meth2 -f pd.DataFrame.spatial.from_geodataframe2 pd.DataFrame.spatial.from_geodataframe2(geo_df)
print(open('lprof_800mb_meth2', 'r').read())


*** Profile printout saved to text file 'lprof_800mb_meth2'. 
Timer unit: 2.93249e-07 s

Total time: 362.825 s
File: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py
Function: from_geodataframe2 at line 2751

Line #      Hits         Time  Per Hit   % Time  Line Contents
  2751                                               @staticmethod
  2752                                               def from_geodataframe2(geo_df):
  2753                                                   """
  2754                                                   Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2755                                                   Requires geopandas library be installed in current environment.
  2756                                           
  2757                                                   =====================  ===============================================================
  2758                                              

### Memory profiling - from_geodataframe()

In [32]:
%mprun -T mprof_800mb_1 -f pd.DataFrame.spatial.from_geodataframe pd.DataFrame.spatial.from_geodataframe(geo_df)
print(open('mprof_800mb_1', 'r').read())



*** Profile printout saved to text file mprof_800mb_1. 
Filename: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py

Line #    Mem usage    Increment   Line Contents
  2700    906.8 MiB    906.8 MiB       @staticmethod
  2701                                 def from_geodataframe(geo_df):
  2702                                     """
  2703                                     Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2704                                     Requires geopandas library be installed in current environment.
  2705                             
  2706                                     =====================  ===============================================================
  2707                                     **Argument**           **Description**
  2708                                     ---------------------  ---------------------------------------------------------------
  2709                                     ge

### Memory profiling - from_geodataframe2()

In [33]:
%mprun -T mprof_800mb_2 -f pd.DataFrame.spatial.from_geodataframe2 pd.DataFrame.spatial.from_geodataframe2(geo_df)
print(open('mprof_800mb_2', 'r').read())



*** Profile printout saved to text file mprof_800mb_2. 
Filename: e:\code\geosaurus\geosaurus\src\arcgis\features\geo\_accessor.py

Line #    Mem usage    Increment   Line Contents
  2751   1401.4 MiB   1401.4 MiB       @staticmethod
  2752                                 def from_geodataframe2(geo_df):
  2753                                     """
  2754                                     Import Geopandas GeoDataFrame into an ArcGIS Spatially enabled DataFrame.
  2755                                     Requires geopandas library be installed in current environment.
  2756                             
  2757                                     =====================  ===============================================================
  2758                                     **Argument**           **Description**
  2759                                     ---------------------  ---------------------------------------------------------------
  2760                                     g